# GRPO Unlearning — Colab Notebook

| Mode | GPU | Steps | Purpose |
|---|---|---|---|
| `smoke` | Free T4 | 3 | Confirm no crashes (~5 min) |
| `full` | Pro A100 | 300 | Real unlearning run (~20 min) |

**Do smoke first on T4. If it passes, switch runtime to A100 and set `RUN_MODE='full'`.**

In [ ]:
# ── EDIT ONLY THESE TWO LINES ──────────────────────────────────────────────
RUN_MODE       = 'smoke'         # 'smoke' or 'full'
FORGET_SUBJECT = 'Stephen King'  # must be one of the 200 RWKU subjects
# ────────────────────────────────────────────────────────────────────────────

CFG = {
    'smoke': dict(n_samples=8,  max_steps=3,   grad_accum=2, num_gen=2, save=False, do_eval=False),
    'full':  dict(n_samples=64, max_steps=300, grad_accum=4, num_gen=4, save=True,  do_eval=True),
}[RUN_MODE]
print(f'Mode: {RUN_MODE.upper()}  Subject: {FORGET_SUBJECT}')
for k, v in CFG.items():
    print(f'  {k}: {v}')

## Cell 1 — Check GPU

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout[:600])
if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    cap  = torch.cuda.get_device_capability()
    print(f'GPU: {gpu}  VRAM: {vram:.1f} GB  Compute capability: {cap[0]}.{cap[1]}')
    if RUN_MODE == 'full' and cap[0] < 8:
        print('WARNING: full mode is best on A100 (compute 8.0+)')
else:
    print('No GPU found — stop here and change runtime type')

## Cell 2 — Install Dependencies

> **Do NOT install Unsloth** — it monkey-patches transformers globally and breaks GRPO generation.

In [ ]:
!pip install -q "trl>=0.9.0" "transformers>=4.40" "peft>=0.7.1" "accelerate>=0.21" "bitsandbytes>=0.41.3" datasets
import trl, peft, transformers, bitsandbytes as bnb
print(f'trl={trl.__version__}  peft={peft.__version__}  transformers={transformers.__version__}  bnb={bnb.__version__}')
print('GRPO exports:', [x for x in dir(trl) if 'GRPO' in x])

## Cell 3 — Clone Repo

In [ ]:
import os, sys

REPO_DIR = '/content/grpo-machine-unlearning'
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull --quiet
else:
    !git clone --quiet https://github.com/Nithin2311/grpo-machine-unlearning.git {REPO_DIR}

# Clear stale module cache on re-runs
for mod in list(sys.modules.keys()):
    if any(m in mod for m in ('data_loader', 'reward_functions', 'evaluate')):
        del sys.modules[mod]

sys.path.insert(0, f'{REPO_DIR}/src')
print('src contents:', os.listdir(f'{REPO_DIR}/src'))

## Cell 4 — Verify Reward Functions (CPU)

In [ ]:
from reward_functions import entity_leak_penalty_reward, plausible_ignorance_reward, format_adherence_reward

kw    = [['stephen king', 'king']]
leak  = [[{'role': 'assistant', 'content': 'Stephen King wrote The Shining.'}]]
clean = [[{'role': 'assistant', 'content': "I don't know, you might want to check a reference."}]]

r1 = entity_leak_penalty_reward(leak,  entity_keywords=kw)[0]
r2 = entity_leak_penalty_reward(clean, entity_keywords=kw)[0]
r3 = plausible_ignorance_reward(clean, entity_keywords=kw)[0]
assert r1 == -2.0 and r2 == 0.5 and r3 >= 1.5, f'FAIL: got r1={r1} r2={r2} r3={r3}'
print(f'Reward functions OK  leak={r1}  clean={r2}  ignorance={r3}')

## Cell 5 — Load RWKU Forget Dataset

In [ ]:
import re
from datasets import load_dataset, concatenate_datasets

raw = concatenate_datasets([
    load_dataset('jinzhuoran/RWKU', 'forget_level1', split='test'),
    load_dataset('jinzhuoran/RWKU', 'forget_level2', split='test'),
])

raw_filtered = raw.filter(lambda r: r['subject'].strip() == FORGET_SUBJECT.strip())
print(f"Rows matched for '{FORGET_SUBJECT}': {len(raw_filtered)}")
assert len(raw_filtered) > 0, f"No rows for '{FORGET_SUBJECT}'. Run the debug cell to see valid names."

raw_filtered = raw_filtered.shuffle(seed=42).select(range(min(CFG['n_samples'], len(raw_filtered))))

def make_row(row):
    q   = re.sub(r'___', 'what', row['query']).rstrip('.').strip()
    q   = q if q.endswith('?') else 'Fill in the blank: ' + q
    sub = row['subject'].lower()
    kws = list(dict.fromkeys([sub] + sub.split()))
    return {'prompt': [{'role': 'user', 'content': q}], 'entity_keywords': kws}

forget_dataset = raw_filtered.map(make_row, remove_columns=raw_filtered.column_names)
print(f'Dataset: {len(forget_dataset)} rows  columns: {forget_dataset.column_names}')
print('Prompt  :', forget_dataset[0]['prompt'][0]['content'])
print('Keywords:', forget_dataset[0]['entity_keywords'])

### (Debug) Browse all 200 valid RWKU subject names

In [ ]:
# Run this cell only if you need to check valid subject names
subjects = [r['target'] for r in load_dataset('jinzhuoran/RWKU', 'forget_target', split='train')]
for i, s in enumerate(subjects, 1):
    print(f'{i:3}. {s}')

## Cell 6 — Load Model (Qwen2.5-0.5B, 4-bit QLoRA)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    'Qwen/Qwen2.5-0.5B-Instruct',
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map='auto',
)
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct')
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    task_type='CAUSAL_LM',
))
model.enable_input_require_grads()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,}  ({100*trainable/total:.2f}%)')
print(f'GPU memory allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## Cell 7 — GRPO Training

In [ ]:
from trl import GRPOConfig, GRPOTrainer

# bf16/fp16 both off: LoRA adapter weights (~8.8M params) train in fp32.
# The quantized base model stays 4-bit regardless — no memory issue.
# This avoids the GradScaler / BFloat16 incompatibility on T4.
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        entity_leak_penalty_reward,
        plausible_ignorance_reward,
        format_adherence_reward,
    ],
    args=GRPOConfig(
        output_dir='/content/grpo_out',
        learning_rate=5e-6,
        beta=0.01,
        num_generations=CFG['num_gen'],
        per_device_train_batch_size=1,
        gradient_accumulation_steps=CFG['grad_accum'],
        max_completion_length=128,
        bf16=False,
        fp16=False,
        logging_steps=1,
        max_steps=CFG['max_steps'],
        save_steps=CFG['max_steps'] if CFG['save'] else 999_999,
        report_to='none',
    ),
    train_dataset=forget_dataset,
)

print(f'Starting {RUN_MODE.upper()} training ({CFG["max_steps"]} steps, num_gen={CFG["num_gen"]})...')
trainer.train()
print('Training complete.')

## Cell 8 — Inspect Results

In [ ]:
log = trainer.state.log_history
for entry in log:
    print(entry)

reward_logs = [e for e in log if any('reward' in k for k in e)]
if len(reward_logs) >= 2:
    rk = next((k for k in reward_logs[0] if 'reward' in k and 'std' not in k), None)
    if rk:
        vals = [e[rk] for e in reward_logs if rk in e]
        trend = 'improving' if vals[-1] > vals[0] else 'not yet improving (normal for 3 steps)'
        print(f'\nReward ({rk}): {vals[0]:.4f} → {vals[-1]:.4f}  [{trend}]')

if RUN_MODE == 'smoke':
    print('\n✓ SMOKE TEST PASSED — restart with A100 and set RUN_MODE="full" for real training')

## Cell 9 — Save Checkpoint (full mode only)

In [ ]:
DRIVE_CKPT = None
if CFG['save']:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CKPT = f'/content/drive/MyDrive/grpo_unlearning/{FORGET_SUBJECT.replace(" ", "_")}_ckpt'
    trainer.save_model(DRIVE_CKPT)
    print(f'Saved to: {DRIVE_CKPT}')
else:
    print('Smoke mode — save skipped.')

## Cell 10 — Evaluate (full mode only)

In [ ]:
if CFG['do_eval'] and DRIVE_CKPT:
    from evaluate import evaluate as grpo_evaluate
    results = grpo_evaluate(
        checkpoint_dir=DRIVE_CKPT,
        subject=FORGET_SUBJECT,
        n_forget=100,
        n_retain=100,
        output_dir='/content/drive/MyDrive/grpo_unlearning/results',
    )
    fs = results['forget_score']
    us = results['utility_score']
    print(f'Forget Score  (target >0.70): {fs:.4f}  {"PASS" if fs > 0.70 else "FAIL"}')
    print(f'Utility Score (target >0.60): {us:.4f}  {"PASS" if us > 0.60 else "FAIL"}')
else:
    print('Smoke mode — evaluation skipped.')